Import Libraries

In [1]:
import torch
import evaluate
import os
import re
import pandas as pd
import numpy as np

from torch import nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

c:\Users\Weedguet\Documents\GitHub\mental_health_classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config

In [ ]:
MODEL_NAME = "distilbert-base-uncased"   # try roberta-base later
MAX_LEN = 32
SEED = 42
EPOCHS = 15
np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

is_multilabel = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

macro_f1 = evaluate.load("f1")
micro_f1 = evaluate.load("f1")

# print(torch.__version__)
# print(torch.version.cuda)   # PyTorch CUDA version
# print(torch.cuda.is_available())

2.7.1+cu118
11.8
True


Toekenization setup

In [3]:
def tok_batch(texts):
    return tokenizer(
        texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
    )

Helper functions

In [4]:
def clean_text(text):
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)
    text = re.sub(r"<[^>]*>", " ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def clean_data(data):
    new_data = data[['label', 'text']].copy()
    texts = new_data['text'].tolist()
    cleaned_texts = [clean_text(str(text)) for text in texts]
    new_data['text'] = cleaned_texts
    return new_data

Metrics

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if not is_multilabel:
        preds = np.argmax(logits, axis=1)
        return {
            "macro_f1": macro_f1.compute(predictions=preds, references=labels, average="macro")["f1"],
            "accuracy": (preds == labels).mean()
        }
    else:
        probs = 1/(1+np.exp(-logits))
        preds = (probs >= 0.5).astype(int)
        return {
            "macro_f1": macro_f1.compute(predictions=preds, references=labels, average="macro")["f1"],
            "micro_f1": micro_f1.compute(predictions=preds, references=labels, average="micro")["f1"]
        }

Inference on new text

In [6]:
def predict(texts, threshold=0.5):
    enc = tok(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = mdl(**enc).logits
    if not is_multilabel:
        ids = torch.argmax(logits, dim=-1).cpu().numpy().tolist()
        return [label_names[i] for i in ids]
    else:
        probs = torch.sigmoid(logits).cpu().numpy()
        pred_mat = (probs >= threshold).astype(int)
        return [[label_names[i] for i,v in enumerate(row) if v==1] for row in pred_mat]

Class for datasets setup

In [7]:
class TextDataset(Dataset):
    def __init__(self, df, is_multilabel=False):
        self.texts = df["text"].tolist()
        if not is_multilabel:
            self.labels = df["y"].tolist()
        else:
            self.labels = [mlb.transform([labs])[0] for labs in df["label"]]
        self.is_multilabel = is_multilabel
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], padding="max_length", truncation=True,
                        max_length=MAX_LEN, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if not self.is_multilabel:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        else:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

Handle class imbalance (Trainer with custom loss)

In [8]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, pos_weight=None, is_multilabel=False, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.pos_weight = pos_weight
        self.is_multilabel = is_multilabel

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels", None)
        outputs = model(**inputs)
        logits = outputs.logits

        if labels is None:
            loss = outputs.loss if hasattr(outputs, "loss") else None
            return (loss, outputs) if return_outputs else loss

        labels = labels.to(logits.device)

        if not self.is_multilabel:
            if self.class_weights is None:
                loss_fct = nn.CrossEntropyLoss()
            else:
                loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = loss_fct(logits, labels)
        else:
            if self.pos_weight is None:
                loss_fct = nn.BCEWithLogitsLoss()
            else:
                loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))
            loss = loss_fct(logits, labels.type_as(logits))

        return (loss, outputs) if return_outputs else loss

Load data

In [ ]:
df = pd.read_csv("mentalhealth_reddit_posts_ds2.csv")

df.rename(columns={"subreddit": "label"}, inplace=True)
df.rename(columns={"selftext": "text"}, inplace=True)

df = clean_data(df)

In [10]:
label_mapping = {
    'anorexia': 'eating_disorder',
    'bulimia': 'eating_disorder',
    'depression': 'mood_disorder',
    'bipolar': 'mood_disorder',
    'anxiety': 'anxiety_disorder',
    'schizophrenia': 'psychotic_disorder',
    'dementia': 'neurocognitive_disorder',
    'borderline': 'personality_disorder'
}

df['label'] = df['label'].replace(label_mapping)


Choose label mode (multi-class OR multi-label)

In [11]:
if not is_multilabel:
    enc = LabelEncoder()
    df["y"] = enc.fit_transform(df["label"])
    label_names = list(enc.classes_)
    num_labels = len(label_names)
    class_counts = df["y"].value_counts().sort_index().to_numpy()
    class_weights = (class_counts.sum() / (num_labels * class_counts))
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    pos_weight = torch.ones(num_labels, dtype=torch.float32).to(device)
else:
    # Ensure label is a list
    if df["label"].dtype != object or not isinstance(df["label"].iloc[0], (list, tuple)):
        raise ValueError("For multi-label, make sure df['label'] is a list of strings per row.")
    mlb = MultiLabelBinarizer()
    Y = mlb.fit_transform(df["label"])
    label_names = list(mlb.classes_)
    num_labels = len(label_names)
    # Positive class weights (for BCEWithLogitsLoss)
    pos_counts = Y.sum(axis=0)
    neg_counts = Y.shape[0] - pos_counts
    pos_weight = torch.tensor(neg_counts / np.clip(pos_counts, 1, None), dtype=torch.float32).to(device)


Train/val/test split

In [12]:
# Split train/test
if not is_multilabel:
    train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["y"], random_state=SEED)
    val_df, test_df  = train_test_split(temp_df, test_size=0.5, stratify=temp_df["y"], random_state=SEED)
else:
    # For multi-label, approximate stratification by frequency (simple random split is common)
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
    val_df, test_df  = train_test_split(temp_df, test_size=0.5, random_state=SEED)

Tokenize the Data & Build datasets

In [13]:
train_ds = TextDataset(train_df, is_multilabel=is_multilabel)
val_ds   = TextDataset(val_df,   is_multilabel=is_multilabel)
test_ds  = TextDataset(test_df,  is_multilabel=is_multilabel)

# Set PyTorch format
# train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
# test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Load Model

In [14]:
if not is_multilabel:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
else:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, problem_type="multi_label_classification"
    )
model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


Training Setup

In [15]:
args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),   # mixed precision on GPU
    logging_steps=50
)

Define Trainer

In [16]:
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,   # from your preprocessing for multi-class
    pos_weight=pos_weight,         # from your preprocessing for multi-label
    is_multilabel=is_multilabel
)

Train the Model

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.676200,1.490676,0.515973,0.525500


TrainOutput(global_step=63, training_loss=1.6449603126162575, metrics={'train_runtime': 251.9166, 'train_samples_per_second': 63.513, 'train_steps_per_second': 0.25, 'total_flos': 132476848128000.0, 'train_loss': 1.6449603126162575, 'epoch': 1.0})

Evaluate

In [18]:
metrics = trainer.evaluate(test_ds)
metrics

{'eval_loss': 1.4972800016403198,
 'eval_macro_f1': 0.49700660075806224,
 'eval_accuracy': 0.516,
 'eval_runtime': 10.8997,
 'eval_samples_per_second': 183.492,
 'eval_steps_per_second': 0.734,
 'epoch': 1.0}

In [19]:
if not is_multilabel:
    preds = np.argmax(trainer.predict(test_ds).predictions, axis=1)
    print(classification_report(test_df["y"], preds, target_names=label_names, digits=3))
else:
    logits = trainer.predict(test_ds).predictions
    probs = 1/(1+np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    print(classification_report(
        np.vstack(test_ds[:]["labels"]), preds, target_names=label_names, digits=3
    ))

                         precision    recall  f1-score   support

       anxiety_disorder      0.460     0.538     0.496       355
        eating_disorder      0.531     0.608     0.567       265
          mood_disorder      0.645     0.456     0.534       691
neurocognitive_disorder      0.586     0.789     0.672       247
   personality_disorder      0.318     0.295     0.306       139
     psychotic_disorder      0.389     0.426     0.406       303

               accuracy                          0.516      2000
              macro avg      0.488     0.519     0.497      2000
           weighted avg      0.528     0.516     0.514      2000



Save & load later

In [20]:
trainer.save_model("./mental_health_classifier")
tokenizer.save_pretrained("./mental_health_classifier")

# Reload
from transformers import AutoModelForSequenceClassification, AutoTokenizer
tok = AutoTokenizer.from_pretrained("./mental_health_classifier")
mdl = AutoModelForSequenceClassification.from_pretrained("./mental_health_classifier").to(device)

Make Predictions

In [21]:
predict(["I’ve been feeling on edge and panicky all week."])

['anxiety_disorder']